## Thorlabs Spectrometer. Data Processing Part 3
### Plot intensity vs power

**Dataset:** `\2026_09_11_spec\p4p5Pa_fast_scan` 

**Steps:** plot figs

In [ ]:
## imports
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
##### import project related moduls ####
current_file = Path.cwd() # cwd = path/*.ipynb - does not work in .py files.
print(f"current_file = {current_file}")
project_root = current_file.parent.parent
print(f"project_root = {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import utils.file_utils as fu   
from utils.plt_styler_avp import PlotStyler

%matplotlib QtAgg

# %load_ext autoreload
# %autoreload 2

In [ ]:
## check data path
folder_path=Path(r"C:\Andrei\DATA\VINETA_75\2026_09_11_spec\p4p5Pa_fast_scan\tables")
# folder_path = None
files = fu.list_files_in_folder(folder_path=folder_path)
list_files = []
print(files[0].parent)
for file in files:
    print(file.name)
    list_files.append(file)

In [ ]:
## load intensity data
print(list_files[0])
print(list_files[1])
spectr_A = pd.read_csv(list_files[0], sep="\t",
                       index_col=0
                       )

spectr_B = pd.read_csv(list_files[1], sep="\t",
                       index_col=0
                       )

# print(spectr_A.head(2))
# print(spectr_B.head(2))
print(spectr_A.info())

In [ ]:
pixel_dict = {
    'Ar2_A': 3611,
    'Ar1_A': 3262,
    'Ar3_A': 3183,
    'Ar4_A': 2478,
    'ArII_A': 1028,

    'Ar2_B': 3611-12, ## pixels shifted for corespondet wavelength on spectrometer B
    'Ar1_B': 3262-11,
    'Ar3_B': 3183-10,
    'Ar4_B': 2478-8,
    'ArII_B': 1028-3,    
}

def get_pixel_data(spec_df, lable='Ar1_A'):
    pixel=str(pixel_dict[lable])
    df = spec_df.copy()
    df = df.sort_index()
    pixel = str(pixel)
    power = df["power"]
    intensity = df[pixel]
    note = lable.split('_')[0]
    return {
        "power": power,
        "intensity": intensity,
        "lable" : note,
    }

def normalize_data(data):
    intensity = data['intensity'].copy()
    power = data['power']
    note = data['lable']
    max_val = intensity.max()
    norm_intensity = intensity / max_val if max_val != 0 else intensity
    return {
            "power": power,
            "intensity": norm_intensity,
            "lable" : note,
        }


In [ ]:
## get intensity data
Ar1_A = get_pixel_data(spectr_A, lable='Ar1_A')
Ar2_A = get_pixel_data(spectr_A, lable='Ar2_A')
ArII_A = get_pixel_data(spectr_A, lable='ArII_A')
Ar3_A =  get_pixel_data(spectr_A, lable='Ar3_A')
Ar4_A =  get_pixel_data(spectr_A, lable='Ar4_A')

Ar1_An = normalize_data(Ar1_A)
Ar2_An = normalize_data(Ar2_A)
ArII_An = normalize_data(ArII_A)
Ar3_An = normalize_data(Ar3_A)
Ar4_An = normalize_data(Ar4_A)

Ar1_B = get_pixel_data(spectr_B, lable='Ar1_B')
Ar2_B = get_pixel_data(spectr_B, lable='Ar2_B')
ArII_B = get_pixel_data(spectr_B, lable='ArII_B')

Ar1_Bn = normalize_data(Ar1_B)
Ar2_Bn = normalize_data(Ar2_B)
ArII_Bn = normalize_data(ArII_B)



In [ ]:
def make_fig(figsize=(10, 5), dpi=100, fontsize=20, fontprofile='compact'):

    styler = PlotStyler()
    styler.set_plt_font_style(size=fontsize, profile=fontprofile)
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    styler.set_scale_steps(ax)

    ax.minorticks_on()
    ax.tick_params(which="major", length=8, width=1.5, direction="out")
    ax.tick_params(which="minor", length=4, width=1.0, direction="out")
    for spine in ax.spines.values():
            spine.set_linewidth(1.5)

    ax.set_xlabel("power [kW]") 
    ax.set_ylabel("intensity [counts/s]") 
    fig.tight_layout()

    return fig, ax

### "tab:blue"   "tab:orange"   "tab:green" "tab:red"  "tab:purple"  "tab:brown"  "tab:pink"
### "tab:gray"   "tab:olive"   "tab:cyan"*
def add_line( ax, data, color="tab:blue", marker="."):
    ax.plot( data['power'], 
        data['intensity'], 
        color=color, 
        marker=marker, 
        linewidth=1.2, 
        label=f"{data['lable']}", 
        )

    ax.legend() 
    ax.figure.canvas.draw_idle()

#### Plot intensity for spectrometer A

In [ ]:
fig_A, ax_A = make_fig(figsize=(8, 6), fontsize=30)
fig_An, ax_An = make_fig(figsize=(8, 6), fontsize=30)

add_line(ax_A, Ar1_A,color="tab:red", marker=".")
add_line(ax_A, Ar2_A,color="tab:orange", marker=".")
add_line(ax_A, ArII_A,color="tab:blue", marker="o")


add_line(ax_An, Ar1_An,color="tab:red", marker=".")
add_line(ax_An, Ar2_An,color="tab:orange", marker=".")
add_line(ax_An, ArII_An,color="tab:blue", marker="o")

#### Plot intensity for spectrometer B

In [ ]:
fig_B, ax_B = make_fig(figsize=(8, 6), fontsize=30)
fig_Bn, ax_Bn = make_fig(figsize=(8, 6), fontsize=30)

add_line(ax_B, Ar1_B,color="tab:red", marker=".")
add_line(ax_B, Ar2_B,color="tab:orange", marker=".")
add_line(ax_B, ArII_B,color="tab:blue", marker="o")


add_line(ax_Bn, Ar1_Bn,color="tab:red", marker=".")
add_line(ax_Bn, Ar2_Bn,color="tab:orange", marker=".")
add_line(ax_Bn, ArII_Bn,color="tab:blue", marker="o")

#### Plot weak argon lines Ar3 and Ar4
spectrometer A 

In [ ]:
fig_As, ax_As = make_fig(figsize=(8, 6), fontsize=30)
fig_Ans, ax_Ans = make_fig(figsize=(8, 6), fontsize=30)

add_line(ax_As, Ar3_A,color="tab:pink", marker=".")
add_line(ax_As, Ar4_A,color="tab:green", marker=".")
add_line(ax_Ans, Ar3_An,color="tab:pink", marker=".")
add_line(ax_Ans, Ar4_An,color="tab:green", marker=".")

### Plot intensity ratios of Ar lines

In [ ]:
def data_ratio(data_base:dict, data_numerator:dict):
    if not np.allclose(data_base["power"], data_numerator["power"]): 
        raise ValueError("Power axes do not match!")
    
    power = data_base["power"].copy()
    ratio = data_numerator["intensity"] / data_base["intensity"] 
    ratio = ratio.replace([np.inf, -np.inf], np.nan)
    lable = f"{data_numerator['lable']} / {data_base['lable']}"
    return {
        "power": power,
        'ratio': ratio,
        'lable': lable,
    }

In [ ]:
Ar2_Ar1_A = data_ratio(Ar1_A, Ar2_A)
Ar2_Ar1_B = data_ratio(Ar1_B, Ar2_B)
ArII_Ar1_A = data_ratio(Ar1_A, ArII_A)
ArII_Ar1_B = data_ratio(Ar1_B, ArII_B)

Ar1B_Ar1A = data_ratio(Ar1_A, Ar1_B)

In [ ]:
def make_fig_ratio(figsize=(10, 5), dpi=100, fontsize=30, fontprofile='compact'):

    styler = PlotStyler()
    styler.set_plt_font_style(size=fontsize, profile=fontprofile)
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    styler.set_scale_steps(ax)

    ax.minorticks_on()
    ax.tick_params(which="major", length=8, width=1.5, direction="out")
    ax.tick_params(which="minor", length=4, width=1.0, direction="out")
    for spine in ax.spines.values():
            spine.set_linewidth(1.5)

    ax.set_xlabel("power [kW]") 
    ax.set_ylabel("intensity ratio") 
    fig.tight_layout()

    return fig, ax

### "tab:blue"   "tab:orange"   "tab:green" "tab:red"  "tab:purple"  "tab:brown"  "tab:pink"
### "tab:gray"   "tab:olive"   "tab:cyan" *
def add_line_ratio( ax, data, color="tab:blue", marker=".", markersize=10):
    ax.plot( data['power'], 
        data['ratio'], 
        color=color, 
        marker=marker, 
        markersize=markersize,
        linewidth=2, 
        label=f"{data['lable']}", 
        )

    ax.legend() 
    ax.figure.canvas.draw_idle()

In [ ]:
fig_r, ax_r = make_fig_ratio(figsize=(8, 6), fontsize=30)

add_line_ratio(ax_r, Ar2_Ar1_A, color="tab:red", marker="." )
add_line_ratio(ax_r, Ar2_Ar1_B, color="tab:orange", marker="." )


In [ ]:
fig_rII, ax_rII = make_fig_ratio(figsize=(8, 6), fontsize=30)
add_line_ratio(ax_rII, ArII_Ar1_A, color="tab:blue" , marker="o" )
add_line_ratio(ax_rII, ArII_Ar1_B, color="tab:cyan", marker="o" )

In [ ]:
fig_BA, ax_BA = make_fig_ratio(figsize=(8, 6), fontsize=30)
add_line_ratio(ax_BA, Ar1B_Ar1A, color="tab:green" , marker="o", markersize=10 )
lines = ax_BA.get_lines()
lines[0].set_label("Ar1_B / Ar1_A")

ax_BA.legend()
ax_BA.figure.canvas.draw_idle()

In [ ]:
ax_r.set_title("argon line ratio (Ar2 / Ar1)")
ax_rII.set_title("ion nuetral ratio (ArII / Ar1)")

In [ ]:
lines = ax_r.get_lines()
print(lines)
for l in lines:
    print(f"{l} color='{l.get_color()}'")

lines[0].set_label("A - antenna")
lines[1].set_label("B - back side")
ax_r.legend()
ax_r.figure.canvas.draw_idle()

In [ ]:
lines = ax_rII.get_lines()
lines[0].set_label("A - antenna")
lines[1].set_label("B - back side")
ax_rII.legend()
ax_rII.figure.canvas.draw_idle()